 Question 5: Meeting Transcript → Action Points
  Task:
  Create a 3-agent pipeline to convert a meeting transcript into actionable tasks.
ListenerAgent: Extracts main discussion points


ActionAgent: Converts them into to-do tasks


DeadlineAgent: Assigns timelines to each task


Agents

🎧 ListenerAgent → Extracts the main discussion points

✅ ActionAgent → Converts discussion into actionable tasks

📅 DeadlineAgent → Assigns realistic deadlines

Install Required Packages

In [1]:
!pip -q install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.8 MB/s eta 0:00:00


Load Groq API

In [2]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq API Loaded Successfully")

✅ Groq API Loaded Successfully


Create the Multi-Agent Pipeline

In [3]:
from groq import Groq

# ---------------------------------------------------
# Base Conversable Agent
# ---------------------------------------------------

class ConversableAgent:

    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt

    def reply(self, message):

        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role":"system",
                    "content":self.system_prompt
                },
                {
                    "role":"user",
                    "content":message
                }
            ],
            temperature=0.3
        )

        return completion.choices[0].message.content.strip()


# ---------------------------------------------------
# Agent 1 : Listener
# ---------------------------------------------------

ListenerAgent = ConversableAgent(
    "ListenerAgent",
    """
You are an expert meeting listener.

Read the meeting transcript.

Extract:

- Main discussion points
- Important decisions
- Key concerns

Return only bullet points.
"""
)

# ---------------------------------------------------
# Agent 2 : Action
# ---------------------------------------------------

ActionAgent = ConversableAgent(
    "ActionAgent",
    """
You are a project manager.

Convert the discussion points into clear actionable tasks.

Return as:

Task 1:
Task 2:
Task 3:

Only actionable items.
"""
)

# ---------------------------------------------------
# Agent 3 : Deadline
# ---------------------------------------------------

DeadlineAgent = ConversableAgent(
    "DeadlineAgent",
    """
You are a delivery manager.

Assign realistic deadlines to every task.

Return as a markdown table.

Task | Deadline

Use realistic timelines like:

Today
Tomorrow
Within 2 Days
This Week
Next Week

Do not invent impossible deadlines.
"""
)

Sample Meeting Transcript

In [4]:
meeting_transcript = """
Project Meeting

John:
The login page still has bugs.

Sarah:
Frontend should fix that before Friday.

Mike:
Backend API documentation is incomplete.

John:
QA team needs test cases.

Sarah:
Client requested dark mode.

Mike:
Deployment should happen next Monday.

Everyone agreed to prepare the demo before deployment.
"""

Run the 3-Agent Workflow

In [5]:
print("="*60)
print("🎧 LISTENER AGENT")
print("="*60)

discussion = ListenerAgent.reply(meeting_transcript)

print(discussion)


print("\n")
print("="*60)
print("✅ ACTION AGENT")
print("="*60)

actions = ActionAgent.reply(discussion)

print(actions)


print("\n")
print("="*60)
print("📅 DEADLINE AGENT")
print("="*60)

deadlines = DeadlineAgent.reply(actions)

print(deadlines)

🎧 LISTENER AGENT
* Main discussion points:
  * Login page bugs
  * Backend API documentation
  * Dark mode request
  * Deployment schedule
* Important decisions:
  * Frontend to fix login page bugs before Friday
  * Deployment scheduled for next Monday
  * Prepare demo before deployment
* Key concerns:
  * Incomplete backend API documentation
  * Need for test cases for QA team
  * Meeting client's dark mode request


✅ ACTION AGENT
Task 1: Fix login page bugs by Friday to ensure a smooth user experience.
Task 2: Complete backend API documentation and create test cases for the QA team to review and test.
Task 3: Prepare a demo of the updated system, including the dark mode feature, to present to the client before next Monday's deployment.


📅 DEADLINE AGENT
| Task | Deadline |
| --- | --- |
| Task 1: Fix login page bugs | This Week |
| Task 2: Complete backend API documentation and create test cases | Next Week |
| Task 3: Prepare a demo of the updated system | This Week |


                Meeting Transcript
                        │
                        ▼
             🎧 ListenerAgent
        (Extract discussion points)
                        │
                        ▼
              ✅ ActionAgent
         (Generate actionable tasks)
                        │
                        ▼
            📅 DeadlineAgent
         (Assign task timelines)
                        │
                        ▼
               Final Action Plan